In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "database").is_dir() and (p / "code").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abra o notebook em Codigos ou em uma de suas subpastas.")
DATABASE_DIR = PROJECT_ROOT / "database"
RESULT_DIR = PROJECT_ROOT / "result" / "estudos"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
# Cada experimento mantém seus próprios dataframes derivados.
PROCESSED_DIR = RESULT_DIR / "processados"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / "code" / "visualizacao"))


<font size="4"><center>UNIVERSIDADE DE SÃO PAULO</center></font>
<font size="4"><center>INSTITUTO DE CIÊNCIAS MATEMÁTICAS E DE COMPUTAÇÃO</center></font>

<font size="4"><center>Núcleo Interinstitucional de Linguística Computacional (NILC)</center></font>

<br>
<font size="3"><center><b>Processamento de Linguagem Natural</b></center></font>

<font size="3"><center>Renato Moraes Silva e Graça Nunes</center></font>
<br/>



# <center>*Word embeddings*</center>

Neste exercício, usaremos técnicas de representação distribuída para representar os textos de uma base de dados de notícias falsas (*fake news*).

### Objetivos de aprendizagem
- Entender como treinar técnicas de representação distribuída no Python
- Entender como usar modelos pré-treinados

### O problema

Veja um exemplo de notícia falsa:
 * ```Polos magnéticos da Terra podem se inverter e causar colapso mundial: "A Terra ficaria inabitável". Aos menos 100 mil pessoas morreriam por ano pela alta nos níveis de radiação espacial." Se o campo magnético continuar a diminuir e os polos magnéticos se inverterem, a Terra pode acabar como Marte – um local seco, árido e incapaz de preservar a vida.```

Exemplo de notícia verdadeira:
 * ```Inversão dos polos magnéticos da Terra pode ocorrer mais rápido do que o previsto. Segundo afirmações, essas ocorrências são, a princípio, indistinguíveis das verdadeiras mudanças nos polos. Apesar dessas reversões não representarem qualquer ameaça à humanidade, os especialistas alertam que poderão gerar falhas nos satélites que orbitam a Terra.```

Faremos o download de uma base de dados de notícias falsas. Essa base pode ser encontrada no seguinte link: <https://github.com/roneysco/Fake.br-Corpus>. Ela foi publicada nos seguintes artigos científicos:

- Renato M. Silva, Roney L. S. Santos, Tiago A. Almeida and Thiago A. S. Pardo  (2020). *Towards Automatically Filtering Fake News in Portuguese*. Expert Systems with Applications, vol 146, p. 113199. DOI: [10.1016/j.eswa.2020.113199](http://dx.doi.org/10.1016/j.eswa.2020.113199).

- Monteiro R.A., Santos R.L.S., Pardo T.A.S., de Almeida T.A., Ruiz E.E.S., Vale O.A. (2018) Contributions to the Study of Fake News in Portuguese: New Corpus and Automatic Detection Results. In: Villavicencio A. et al. (eds) Computational Processing of the Portuguese Language. PROPOR 2018. Lecture Notes in Computer Science, vol 11122. Springer.

Primeiro, vamos importar as bibliotecas que serão usadas neste exercício.

In [4]:
import numpy as np # biblioteca usada para trabalhar com vetores e matrizes
import pandas as pd # biblioteca usada para trabalhar com dataframes e análise de dados
import re # biblioteca para expressoes regulares
import os # biblioteca usada para realizar tarefas específicas ao SO

from zipfile import ZipFile # biblioteca para arquivos zipados

Vamos baixar uma base de dados.

Vamos descompactar o arquivo que contém a base de dados.

In [5]:
import os
import subprocess

def download(url, path):

    # comando para fazer o download
    cmd = ['wget', '-q', url, '-O', '%s' %(path)]

    # executa o comando sem exibir a saída na tela
    processo = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # aguarda o término do download
    stdout, stderr = processo.communicate()

    # Verifica se o processo foi concluído com sucesso (código de retorno zero)
    if processo.returncode == 0:
        print(f'Processo finalizado.')
    else:
        print(f'Houve um erro: {stderr.decode("utf-8")}')

# especifica o local onde ficarao os arquivos
pathFiles = str(PROJECT_ROOT / 'code' / 'apoio' / 'recursos') + '/'

# cria uma pasta onde ficarao os arquivos
if not os.path.isdir(pathFiles):
    os.mkdir(pathFiles)

url = 'https://github.com/roneysco/Fake.br-Corpus/archive/refs/heads/master.zip'
pathDataset1 = pathFiles + 'Fake.br-Corpus-master.zip'
download(url, pathDataset1)

Processo finalizado.


In [6]:
def unzip(path, pathFolder):

    try:

        # descompacta o arquivo
        z = ZipFile(path, 'r')
        z.extractall(pathFolder)
        z.close()

        print("Arquivo descompactado com sucesso!")

    except:
        print("Houve um erro ao tentar descompactar o arquivo")

unzip(pathDataset1, pathFiles)

Arquivo descompactado com sucesso!


Vamos importar os dados da base.

In [7]:
def import_fakeNewsDataset(path):

    files = os.listdir(path)

    dataset = []

    for i, file in enumerate(files):

        # abre o arquivo
        fileTexto = open(path+'/'+file, "r", encoding='utf8', errors='replace') #utf8 #cp1252

        # le o arquivo e armazena o texto dentro da variavel "texto"
        texto = fileTexto.read()

        dataset.append(texto)

    return dataset

# importa a base de dados de noticias falsas
dataset_fake = import_fakeNewsDataset(str(DATABASE_DIR / 'Fake.br' / 'corpus' / 'full_texts' / 'fake'))
print('Noticias falsas importadas com sucesso')

# cria um vetor de 1 para indicar a classe de cada noticia falsa
target_fake = np.ones( len(dataset_fake), dtype=int )

# importa a base de dados de noticias falsas
dataset_true = import_fakeNewsDataset(str(DATABASE_DIR / 'Fake.br' / 'corpus' / 'full_texts' / 'true'))
print('Noticias verdadeiras importadas com sucesso')

# cria um vetor de 0 para indicar a classe de cada noticia verdadeira
target_true = np.zeros( len(dataset_true), dtype=int )

# concatena as noticias verdadeiras e falsas
dataset = np.asarray( dataset_fake+dataset_true )

# concatena os vetores de classes das notícias verdadeiras e falsas
target = np.concatenate((target_fake,target_true))

# obtem as classes do problema
classes = np.unique(target)

print('\nQtd. de noticias falsas: %d' %len(dataset_fake))
print('Qtd. de noticias verdadeiras: %d' %len(dataset_true))

Noticias falsas importadas com sucesso
Noticias verdadeiras importadas com sucesso

Qtd. de noticias falsas: 3600
Qtd. de noticias verdadeiras: 3600


Agora, vamos tratar os textos importados da base de dados. Como as notícias foram escritas em língua portuguesa, devemos usar uma função de estemização apropriada para a língua portuguesa. Iremos também remover os acentos das palavras.

Outro ponto importante é que as notícias verdadeiras em geral são mais longas que as notícias falsas. Então, para evitar que a predição do classificador seja mais influenciada pelo tamanho da notícia do que pelo conteúdo, na função de pré-processamento, o texto será truncado para apenas 200 termos.

In [8]:
%pip install nltk

import nltk
import unicodedata # sera usada para remover acentos dos documentos em lingua portuguesa

# Download the stopwords corpus
nltk.download('stopwords')

# Download the RSLPStemmer
nltk.download('rslp')

from nltk.stem import RSLPStemmer # para fazer a estemização em documentos da lingua portuguesa

def preprocessing_portuguese(text, stemming = False, stopwords = False):
    """
    Funcao usada para tratar textos escritos na lingua portuguesa

    Parametros:
        text: variavel do tipo string que contem o texto que devera ser tratado

        stemming: variavel do tipo boolean que indica se a estemizacao deve ser aplicada ou nao

        stopwords: variavel do tipo boolean que indica se as stopwords devem ser removidas ou nao
    """

    ########################## COMPLETE O CÓDIGO AQUI  ########################

    # Lower case
    text = text.lower()

    # remove os acentos das palavras
    nfkd_form = unicodedata.normalize('NFKD', text)
    text = u"".join([c for c in nfkd_form if not unicodedata.combining(c)])

    # remove tags HTML
    regex = re.compile('<[^<>]+>')
    text = re.sub(regex, " ", text)

    # normaliza as URLs
    regex = re.compile('(http|https)://[^\s]*')
    text = re.sub(regex, "<URL>", text)

    # normaliza emails
    regex = re.compile('[^\s]+@[^\s]+')
    text = re.sub(regex, "<EMAIL>", text)

    # converte todos os caracteres não-alfanuméricos em espaço
    regex = re.compile('[^A-Za-z0-9]+')
    text = re.sub(regex, " ", text)

    # normaliza os numeros
    regex = re.compile('[0-9]+.[0-9]+')
    text = re.sub(regex, "NUMERO", text)

    # normaliza os numeros
    regex = re.compile('[0-9]+,[0-9]+')
    text = re.sub(regex, "NUMERO", text)

    # normaliza os numeros
    regex = re.compile('[0-9]+')
    text = re.sub(regex, "NUMERO", text)


    # substitui varios espaçamentos seguidos em um só
    text = ' '.join(text.split())

    # separa o texto em palavras
    words = text.split()

    # trunca o texto para apenas 200 termos
    words = words[0:200]

    # remove stopwords
    if stopwords:
        words = text.split() # separa o texto em palavras
        words = [w for w in words if not w in nltk.corpus.stopwords.words('portuguese')]
        text = " ".join( words )

    # aplica estemização
    if stemming:
        stemmer_method = RSLPStemmer()
        words = [ stemmer_method.stem(w) for w in words ]
        text = " ".join( words )

    # remove palavras compostas por apenas um caracter
    words = text.split() # separa o texto em palavras
    words = [ w for w in words if len(w)>1 ]
    text = " ".join( words )

    return text

exemplo_noticia = 'Quase 30 anos depois, banhistas assustados estão se afastando do principal balneário de Uruguaiana, na Fronteira Oeste. '
print('Antes do preprocessamento: \n', exemplo_noticia)

# executa a função de pré-processsamento para tratar a amostra de texto
exemplo_noticia = preprocessing_portuguese(exemplo_noticia, stemming = True, stopwords = True)

print('\nDepois do preprocessamento: \n', exemplo_noticia)

<>:40: SyntaxWarning: invalid escape sequence '\s'
<>:44: SyntaxWarning: invalid escape sequence '\s'
<>:40: SyntaxWarning: invalid escape sequence '\s'
<>:44: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_12453/2271790213.py:40: SyntaxWarning: invalid escape sequence '\s'
  regex = re.compile('(http|https)://[^\s]*')
/tmp/ipykernel_12453/2271790213.py:44: SyntaxWarning: invalid escape sequence '\s'
  regex = re.compile('[^\s]+@[^\s]+')


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Antes do preprocessamento: 
 Quase 30 anos depois, banhistas assustados estão se afastando do principal balneário de Uruguaiana, na Fronteira Oeste. 

Depois do preprocessamento: 
 quas numer ano banh assust esta afast princip balneari uruguai fronteir oest


[nltk_data] Downloading package stopwords to /home/joao-
[nltk_data]     lucas/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /home/joao-lucas/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


Iremos aplicar a pré-processamento em todos os documentos. Depois, precisaremos converter cada documento para uma lista de palavras.

In [9]:
dataset2 = []
for i, msg in enumerate(dataset):

    msg = preprocessing_portuguese( msg )
    msg = msg.split()

    dataset2.append(msg)

print("\n\nPrimeira amostra")
print(dataset2[10])



Primeira amostra
['stf', 'podera', 'anular', 'decisao', 'do', 'embusteiro', 'trapalhao', 'impeachment', 'nao', 'sera', 'anulado', 'deputado', 'carlos', 'sampaio', 'disse', 'que', 'oposicao', 'ira', 'entrar', 'imediatamente', 'com', 'um', 'mandado', 'de', 'seguranca', 'no', 'stf', 'contra', 'decisao', 'de', 'waldir', 'maranhao', 'que', 'tenta', 'anular', 'votacao', 'do', 'impeachment', 'de', 'dilma', 'sampaio', 'disse', 'que', 'essa', 'uma', 'decisao', 'maluca', 'estapafurdia', 'ele', 'quer', 'anular', 'unilateralnente', 'um', 'ato', 'juridico', 'finalizado', 'pela', 'camara', 'em', 'plenario', 'em', 'rito', 'definido', 'pelo', 'supremo', 'lider', 'do', 'dem', 'pauderney', 'avelino', 'dem', 'am', 'disse', 'que', 'waldir', 'maranhao', 'nao', 'tem', 'capacidade', 'de', 'anular', 'qualquer', 'ato', 'que', 'ja', 'passou', 'pela', 'camara', 'quero', 'tranquilizar', 'populacao', 'porque', 'esse', 'ato', 'nao', 'ira', 'prosperar', 'disse', 'lider', 'do', 'dem', 'presidente', 'do', 'senado', 

---
## Gerando word embeddings

Depois de fazer o pré-processamento, é necessário transformar o texto em um vetor de atributos com valores numéricos. Podemos fazer isso usando word embeddings pré-treinadas ou treinando um modelo próprio.

Neste momento, vamos treinar modelo próprio de embeddings baseado nos dados. Para manipular as embeddings, iremos usar a biblioteca Gensim: https://radimrehurek.com/gensim/models/word2vec.html

In [11]:
# instala a biblioteca
%pip install gensim --quiet

# importa
import gensim
from gensim.models import Word2Vec

########################## COMPLETE O CÓDIGO AQUI  ########################

embeddingModel = Word2Vec(sentences = dataset2,
                          vector_size = 200,
                          window = 3,
                          min_count = 1)

vocabSize = len(embeddingModel.wv)

##########################################################################

print("\nTamanho do vocabulário do modelo: ", vocabSize)

Note: you may need to restart the kernel to use updated packages.


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'



Tamanho do vocabulário do modelo:  80782


O código abaixo apresenta os termos mais próximas de uma determinada termos com base similaridade de coseno.  

In [12]:
########################## COMPLETE O CÓDIGO AQUI  ########################

palavraAlvo = "politico"
similares = embeddingModel.wv.most_similar(palavraAlvo,
                                           topn = 10)

for sim in similares:
    print(sim)

##########################################################################

('juridico', 0.7042257785797119)
('economico', 0.6906896829605103)
('ideologico', 0.6417044401168823)
('unico', 0.6371291875839233)
('partidario', 0.6127830147743225)
('financeiro', 0.5942599177360535)
('inimigo', 0.590333878993988)
('eleitor', 0.5869652032852173)
('construtivo', 0.5823372006416321)
('corrupto', 0.5817165374755859)


---
## Gerando vetores para os documentos

Nós treinamos um modelo que encontra um vetor de representação para cada palavra do vocabulário. No entanto, nós queremos também gerar um vetor para o documento. Para fazer isso, geralmente é feita a soma ou a média dos vetores das palavras do documento. O código abaixo cria o vetor para o documento.

In [13]:
def getDocvector(model, doc):
    """
    obtem o vetor de cada palavra de um documento e calcula um vetor medio
    """

    ########################## COMPLETE O CÓDIGO AQUI  ########################

    wordList = []
    for word in doc:

        try:
            vec = model.wv[word]
            wordList.append(vec)
        except:
            pass

    if len(wordList)>0:
        vetorMedio = np.mean( wordList, axis=0 )
    else:
        vetorMedio = np.zeros( model.we.vector_size )

    ##########################################################################

    return vetorMedio

# vamos testar em uma frase
doc = ['eu', 'tenho', 'um', 'carro', 'amarelo']

docVector = getDocvector(embeddingModel, doc)
print(docVector)

[-1.04708217e-01 -9.22965556e-02 -1.90992951e-02 -6.08452633e-02
  5.61543822e-01  4.47359800e-01 -6.49319410e-01  8.52973104e-01
  1.28612192e-02  1.02419719e-01 -1.08073451e-01 -4.28056002e-01
  8.03692162e-01 -1.48370102e-01  6.96699992e-02  1.07403375e-01
 -7.47937620e-01 -9.90067497e-02  2.47551918e-01 -4.05809730e-01
 -1.66673865e-02  3.61876492e-03  2.61716574e-01  4.69973654e-01
  2.90861368e-01 -3.66935313e-01  4.46139038e-01 -8.93203080e-01
  1.80019215e-01 -4.95220460e-02 -1.13121819e+00  6.01601958e-01
  3.80664289e-01 -1.42585814e-01  1.28107607e-01 -2.45800570e-01
  7.33769000e-01  2.69670337e-01  1.77399918e-01 -8.90813231e-01
 -1.58966452e-01  4.95878980e-02  1.28459558e-01  4.95971262e-01
  5.41956067e-01 -5.72248325e-02 -5.82697213e-01  9.48342830e-02
  4.80088234e-01 -3.74069452e-01 -2.01723188e-01 -2.35613063e-01
 -2.66586840e-02  2.04421803e-01  2.03784239e-02 -2.48077899e-01
 -4.51947361e-01 -2.65979350e-01 -7.58137345e-01 -3.34218770e-01
 -7.05582142e-01  5.73875

Vamos gerar vetores para todos os documentos da base de dados

In [16]:
########################## COMPLETE O CÓDIGO AQUI  ########################

X = []
for doc in dataset2:
    vec = getDocvector(embeddingModel, doc)
    X.append(vec)


X = np.array(X)
print(X.shape)

##########################################################################

[[-0.10565846 -0.11931019  0.27075595 ... -0.09802768 -0.06715885
  -0.24121931]
 [-0.1640993   0.01996373  0.19905439 ... -0.17335236 -0.01974336
  -0.34751606]
 [-0.23556797 -0.14304723  0.22238487 ... -0.19637373 -0.01221322
  -0.31694233]
 ...
 [-0.20750573 -0.0395024   0.3063615  ... -0.3018917   0.01338278
  -0.31184396]
 [-0.15172549 -0.08603189  0.26441756 ... -0.16106153  0.0851481
  -0.04357702]
 [-0.26095802 -0.06885421  0.3319565  ... -0.26367596  0.00475002
  -0.20012125]]
(7200, 200)


Busque os dois documentos de treinamento mais próximos a um determinado texto.

In [18]:

import sklearn as skl
from sklearn import metrics

text = "A nova guerra nuclear começou. Estamos todos em perigo!!!"

########################## COMPLETE O CÓDIGO AQUI  ########################

text = preprocessing_portuguese(text)

# converter para lista
text = text.split()

vecText = getDocvector( embeddingModel, text)

vecText = np.array( [vecText] )

cosenoSim = skl.metrics.pairwise.cosine_similarity(X,
                                            Y = vecText)
cosenoSim = cosenoSim[:,0]

idxOrd = np.argsort(cosenoSim)[::-1]

for idx in idxOrd[0:3]:
    print("\n\n", dataset[idx])

##########################################################################

Defaulting to user installation because normal site-packages is not writeable
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 7.3 MB/s eta 0:00:0000:0100:01
Using cached narwhals-2.22.1-py3-none-any.whl (454 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


 Pentágono diz que Guerra Mundial é inevitável e quase garantida: "Será rápida e letal".  "Um conflito em um futuro próximo será letal e rápido [...] e não conseguiremos cronometrá-lo".  O Major-General William Hix disse: "O número de mortos poderá aumentar exponencialmente. A velocidade dos acontecimentos poderão ultrapassar nossa capacidade humana de acompanhá-la." O comandante da Marinha Gen Joseph Dunford, declarou que os EUA devem "declarar guerra à Síria e Rússia", de modo que uma "zona de exclusão aérea" sobr

---
### Usando modelos pré-treinados

Em vez de treinar um modelo Word2Vec do zero, você pode usar um modelo pré-treinado, como por exemplo o modelo treinado na Google News: https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing

Para usar o modelo pré-treinado, você pode usar o seguinte código:

```
from gensim.models import KeyedVectors
from gensim import models

word2vec_path = 'path/GoogleNews-vectors-negative300.bin.gz'
w2v_model = models.KeyedVectors.load_word2vec_format(word2vec_path, binary=True)
```

Para a lingua portuguesa também existem vários modelos pré-treinados como os oferecidos no seguinte link: <http://nilc.icmc.usp.br/nilc/index.php/repositorio-de-word-embeddings-do-nilc>.

Vamos baixar o modelo do FastText-skipgram de 50 dimensões.

In [22]:

import gdown

url = "https://drive.google.com/file/d/1dAbEJ0psG-r4lbxAuGOYIv5Vu3mQj6PY/view?usp=sharing"

pathFiles = str(PROJECT_ROOT / 'code' / 'apoio' / 'recursos') + '/'
pathDataset2 = pathFiles + 'skip_s50.zip'

# download a partir de um link do GDrive
gdown.download(url, pathDataset2)

Downloading...
From (original): https://drive.google.com/uc?id=1dAbEJ0psG-r4lbxAuGOYIv5Vu3mQj6PY
From (redirected): https://drive.google.com/uc?id=1dAbEJ0psG-r4lbxAuGOYIv5Vu3mQj6PY&confirm=t&uuid=0d3e85d2-7e79-4665-945f-856dc0d8bcc9
To: /home/joao-lucas/Área de trabalho/Projetos/IC/Codigos/dados/skip_s50.zip
100%|██████████| 163M/163M [00:06<00:00, 25.7MB/s] 


'dados/skip_s50.zip'

Vamos descompactar o arquivo

In [23]:
unzip(pathDataset2, pathFiles)

Arquivo descompactado com sucesso!


Carregue o modelo usando a biblioteca Gensim e mostre o tamanho do vocabulário.

In [24]:
import gensim
from gensim.models import KeyedVectors
from gensim import models

########################## COMPLETE O CÓDIGO AQUI  ########################

embeddingModel2 = gensim.models.KeyedVectors.load_word2vec_format(str(PROJECT_ROOT / 'code' / 'apoio' / 'recursos' / 'skip_s50.txt'))

# Obter o tamanho do vocabulário do modelo
vocabSize = len(embeddingModel2.index_to_key)
print("Tamanho do vocabulário do modelo: ", vocabSize)

##########################################################################

Tamanho do vocabulário do modelo:  929605


Vamos dar uma olhada na similaridade de coseno de algumas palavras do modelo.

In [25]:
########################## COMPLETE O CÓDIGO AQUI  ########################

sims = embeddingModel2.most_similar('carro', topn=10)  # get other similar words
for sim in sims:
    print(sim)

##########################################################################

('caminhão', 0.9377034306526184)
('furgão', 0.8873354196548462)
('camionete', 0.8857896327972412)
('bagageiro', 0.874361515045166)
('caminhã', 0.8735042214393616)
('porta-malas', 0.8730326890945435)
('carrro', 0.8680179715156555)
('caminhã£o', 0.8669916987419128)
('caminhãozinho', 0.8668836951255798)
('caminhonete', 0.8657148480415344)
